# Strava API thingy

- See the [Getting Started Docs](https://developers.strava.com/docs/getting-started/)
- You can get data on yourself without authentication for testing purposes.
- See also the [Strava API v3 reference docs](https://developers.strava.com/docs/reference/)

See [README.md](README.md) for more information about what I want to achive here.


## Contents

- <a href="#Authentication">Authentication</a>
- <a href="#Basic-API-Requests">Basic API Requests</a>
- <a href="#Pandas-Data-Frames">Pandas Data Frames</a>
- <a href="#Data-Streams">Data Streams</a> -- i.e. watts, cadence, heart rate, time, etc.

----

In [1]:
import os
import requests
import pandas as pd

client_id = os.environ.get("CLIENT_ID")
client_secret = os.environ.get("CLIENT_SECRET")
access_token = os.environ.get("ACCESS_TOKEN")

In [2]:
class StravaClient:
    BASE_URL = "https://www.strava.com/api/v3"

    def __init__(self, client_id, client_secret, access_token=None, refresh_token=None):
        self.client_id = client_id
        self.client_secret = client_secret
        self.access_token = access_token
        self.refresh_token = refresh_token

    @property
    def _headers(self):
        return {"Authorization": f"Bearer {self.access_token}"}

    def set_token_from_response(self, data: dict):
        self.access_token = data["access_token"]
        self.refresh_token = data.get("refresh_token")

    def exchange_code(self, code: str) -> dict:
        """Exchange authorization CODE for tokens (first-time OAuth flow)."""
        url = "https://www.strava.com/oauth/token"
        payload = {
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "code": code,
            "grant_type": "authorization_code",
        }
        resp = requests.post(url, payload)
        resp.raise_for_status()
        self.set_token_from_response(resp.json())
        return resp.json()

    def refresh_access_token(self) -> dict:
        """Use refresh_token to get a new access_token (call when token expires)."""
        url = "https://www.strava.com/oauth/token"
        payload = {
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "grant_type": "refresh_token",
            "refresh_token": self.refresh_token,
        }
        resp = requests.post(url, payload)
        resp.raise_for_status()
        self.set_token_from_response(resp.json())
        return resp.json()

    def get_athlete(self) -> dict:
        """Fetch the authenticated athlete's profile."""
        resp = requests.get(f"{self.BASE_URL}/athlete", headers=self._headers)
        resp.raise_for_status()
        return resp.json()

    def get_activities(self, per_page: int = 100) -> list:
        """Fetch ALL activities using pagination (loops until empty page)."""
        all_activities = []
        page = 1
        while True:
            resp = requests.get(
                f"{self.BASE_URL}/athlete/activities",
                headers=self._headers,
                params={"per_page": per_page, "page": page},
            )
            resp.raise_for_status()
            batch = resp.json()
            if not batch:
                break
            all_activities.extend(batch)
            page += 1
        return all_activities

    def get_activity_streams(self, activity_id: int, stream_types: list) -> list:
        """Fetch data streams for a single activity."""
        resp = requests.get(
            f"{self.BASE_URL}/activities/{activity_id}/streams",
            headers=self._headers,
            params={"keys": ",".join(stream_types), "key_by_type": ""},
        )
        resp.raise_for_status()
        return resp.json()

---- 

## Authentication

In [3]:
# Instantiate the client (uses token from env if available)
client = StravaClient(client_id, client_secret, access_token=access_token)
token_status = f"set ({client.access_token[:10]}...)" if client.access_token else "not set — run the OAuth cell below"
print(f"Client ready. Token: {token_status}")

In [ ]:
# Run this cell to authorize via browser and get a fresh token.
# Prerequisite: set redirect URI to "http://localhost:8080/exchange_token" in your Strava app settings.
# https://www.strava.com/settings/api
#
# This cell blocks until Strava redirects back — just click Authorize in the browser.

import threading
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse, parse_qs

PORT = 8080

class _OAuthHandler(BaseHTTPRequestHandler):
    code = None

    def do_GET(self):
        parsed = urlparse(self.path)
        params = parse_qs(parsed.query)
        if "code" in params:
            _OAuthHandler.code = params["code"][0]
            self.send_response(200)
            self.end_headers()
            self.wfile.write(b"<h2>Authorization successful!</h2><p>You can close this tab.</p>")
        else:
            self.send_response(400)
            self.end_headers()
            self.wfile.write(b"Authorization failed: no code in redirect.")
        threading.Thread(target=self.server.shutdown).start()

    def log_message(self, format, *args):
        pass  # Suppress server request logs

scopes = "read,read_all,activity:read,activity:read_all"
redirect_uri = f"http://localhost:{PORT}/exchange_token"
auth_url = (
    f"http://www.strava.com/oauth/authorize?client_id={client_id}"
    f"&response_type=code&redirect_uri={redirect_uri}"
    f"&approval_prompt=force&scope={scopes}"
)

server = HTTPServer(("localhost", PORT), _OAuthHandler)
print(f"Opening browser for Strava authorization (waiting on port {PORT})...")
webbrowser.open(auth_url)
server.serve_forever()  # Blocks until the redirect is caught

if _OAuthHandler.code:
    client.exchange_code(_OAuthHandler.code)
    print(f"Done. access_token: {client.access_token[:10]}...")
else:
    print("No code received — authorization may have been cancelled.")

In [6]:
# --- Token Refresh (run this if you get 401 errors — tokens expire after 6 hours)
# The refresh_token from the exchange_code response is stored in client.refresh_token.
# Uncomment and run to get a fresh access_token without going through the browser flow.

data = client.refresh_access_token()
print(f"New access_token expires at: {data['expires_at']}")

HTTPError: 401 Client Error: Unauthorized for url: https://www.strava.com/oauth/token

---- 

## Basic API Requests

In [7]:
# Basic Requests
athlete = client.get_athlete()
athlete

HTTPError: 401 Client Error: Unauthorized for url: https://www.strava.com/api/v3/athlete

In [8]:
# Fetches ALL activities across all pages (may take several seconds for large accounts)
activities = client.get_activities()
print(f"Found {len(activities)} total activities.")

HTTPError: 401 Client Error: Unauthorized for url: https://www.strava.com/api/v3/athlete/activities?per_page=100&page=1

In [9]:
# Activity fields
sorted(activities[0].keys())

NameError: name 'activities' is not defined

In [10]:
for activity in activities:
    dt = activity["start_date_local"].split("T")[0]
    print(f"[{activity['id']}|{dt}] {activity['name']}")

NameError: name 'activities' is not defined

---- 

## Pandas Data Frames

In [11]:
def extract(data: dict, keys: list) -> dict:
    """Extract a dict containing the given keys from an input dict"""
    # See: https://stackoverflow.com/a/74383892/182778
    return {key: val for key in keys if (val := data.get(key))}

KEYS = [
    "name",
    "id",
    "resource_state",  # 2 is cycling
    "start_date",
    "moving_time",
    'average_cadence',
    'average_heartrate',
    'average_speed',
    'average_watts',
    'max_heartrate',
    'max_speed',
    'max_watts',
    'weighted_average_watts',
    'suffer_score',
    'kilojoules',
    'distance',
    'elapsed_time',
    'elev_high',
    'elev_low',
]


df = pd.DataFrame([extract(a, KEYS) for a in activities], columns=KEYS)
df = df.set_index("start_date")
df.index = pd.to_datetime(df.index, utc=True)
df.head()

NameError: name 'activities' is not defined

In [12]:
df[["average_watts","average_heartrate","max_heartrate","suffer_score"]].plot(rot=45)

NameError: name 'df' is not defined

In [13]:
df[['elev_low', 'elev_high']].plot(rot=45)

NameError: name 'df' is not defined

In [14]:
df[['kilojoules', 'moving_time']].plot(rot=45)

NameError: name 'df' is not defined

In [15]:
df[["average_heartrate","max_heartrate"]].plot(rot=45)


NameError: name 'df' is not defined

----

## Data Streams

https://developers.strava.com/docs/reference/#api-Streams-getActivityStreams

> Returns the given activity's streams. Requires activity:read scope. Requires activity:read_all scope for Only Me activities.


See also: Strava Docs on `StreamSet` -- types of data available in streams.
https://developers.strava.com/docs/reference/#api-models-StreamSet

In [16]:
# Use the most recent activity, or pin a specific one below
activity_id = activities[0]["id"]
# activity_id = 10804601892

STREAM_TYPES = ["time", "distance", "heartrate", "cadence", "watts"]
streams = client.get_activity_streams(activity_id, STREAM_TYPES)
print(f"Found {len(streams)} streams for activity {activity_id}.")

# --- Stream object keys
for i, stream in enumerate(streams):
    stype = stream["type"]
    print(f"Stream {i}: {stype}: {', '.join(stream.keys())}")

NameError: name 'activities' is not defined

In [17]:
for i, stream in enumerate(streams):
    print(stream['type'], stream['series_type'], stream['original_size'], stream['resolution'])

NameError: name 'streams' is not defined

In [18]:
# Create a Data Frame with all the data streams
ds = {s['type']: pd.Series(s['data']) for s in streams}
df = pd.DataFrame(ds).set_index('time')
df.head()

NameError: name 'streams' is not defined

In [19]:
df[["cadence", "watts", "heartrate"]].plot()

NameError: name 'df' is not defined